# Objectif 

Sélection des données à traiter.

Ces données sont en libre accès et directement interrogeables sur la plateforme EarthScope (anciennement IRIS). Pour plus d’informations sur les réseaux CY30x et CY60x, je vous invite à consulter la page dédiée : https://ds.iris.edu/mda/IM/.

In [1]:
# this will download 1 hour of data from a FR network station, and display a wafeform plot.
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import scipy.io.wavfile as wavfile
import scipy.signal as sp 

from obspy import UTCDateTime
from obspy import read as obspy_read
from obspy import read_inventory as obspy_read_inventory
from obspy import Stream, Inventory
from obspy.clients.fdsn import Client

from pyproj import Geod
from scipy import signal
from matplotlib import pyplot as plt


sys.path.append(r"C:\Users\baptiste.menetrier\Desktop\devPy\phd")
from source.bruit_fm_manager import BruitfmManager

from publication.publication_figure import LargeFigure

import pandas as pd
from stage_M1.utils import process_and_plot

In [2]:
root_data = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\reseau_9R\data"
)

# Get all stations available on bruit-fm

In [3]:
# available_clients = ["IRIS", "RESIF"]
available_clients = ["RESIF"]

networks = ["1B", "1E", "IM"]
root_csv_path = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\data\wav\bruit_fm\stations"
)
fpath_all_stations = os.path.join(root_csv_path, "bruit_fm_stations.csv")

use_client_id = 0

geod = Geod(ellps="WGS84")
# client = Client(available_clients[use_client_id])

# Flags
# parse_networks = True
scrap_all_stations = False
merge_all_stations = False
compute_elevation_diff_gebco = False

In [4]:
bruit_fm_manager = BruitfmManager(
    root_station_storage=root_csv_path, available_clients=available_clients
)

In [5]:
# if parse_networks:
# #     # Parse all network names from html file
#     bruit_fm_manager.parse_stations_from_html()
if scrap_all_stations:
    # Scrap all stations from selected networks
    bruit_fm_manager.scrap_networks_stations()
if merge_all_stations:
    # Merge all stations from selected networks
    bruit_fm_manager.merge_scraped_networks()
if compute_elevation_diff_gebco:
    # Compute elevation difference between stations and GEBOD topography
    bruit_fm_manager.compute_elevation_diff_gebco()

## Apply filter to get the stations matching requirements 

In [6]:
# Filter stations
required_channels = ["BDH", "EDH"]  # "Z", "BDH", "EDH",  "HHZ", "EHZ"
min_recording_duration = 7 * 24 * 60 * 60  # in seconds
max_distance_between_stations = 5  # in km -> corresponds to 100 x lambda for f = 50 Hz
selected_year = 2022
min_nb_rcv_by_array = 2  # Minimum number of receivers by array (grouped stations after distance selection)
min_depth = 0  # in meters -> to meet the deep water assumption

filtered_stations_fname = "hydro_on_seafloor"
filters = {
    "restricted_status": "open",
    "required_channels": required_channels,
    "min_recording_duration": min_recording_duration,
    "max_distance_between_stations": max_distance_between_stations,
    "selected_year": selected_year,
    "min_nb_rcv_by_array": min_nb_rcv_by_array,
    "min_depth": min_depth,
    "receiver_on_seafloor": True,
    "receiver_on_seafloor_tolerance": 100,
}
filtered_stations, _ = bruit_fm_manager.filter_stations(
    filters, save=True, verbose=False, filtered_stations_fname=filtered_stations_fname
)

================ Selected stations stats ================
Number of selected stations : 20
Number of networks : 3
Number of arrays (grouped stations) : 3
Number of stations in network 8A : 6
Number of stations in network IM : 4
Number of stations in network 9R : 10


In [7]:
ds_all_stations = pd.read_csv(bruit_fm_manager.all_stations_fpath)
ds_all_stations = bruit_fm_manager.pre_process_stations(ds_all_stations)

In [8]:
ds_9R = ds_all_stations[ds_all_stations["network"] == "9R"]
ds_9R

,client,network,station,restricted_status,latitude,longitude,elevation,channels,start_date,end_date,duration,height_above_mean_sea_level,elevation_diff
57523,IRIS,9R,OBS01,open,18.51055,-81.76852,-5114.0,EDH_EL1_EL2_ELZ,2022-12-05,2023-06-23 23:59:59.999900,1.736640e+07,-5102,-12.0
57524,IRIS,9R,OBS02,open,18.51072,-81.74122,-4533.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-23 23:59:59.999900,1.728000e+07,-4587,54.0
57625,IRIS,9R,OBS03,open,18.48431,-81.76879,-5139.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-23 23:59:59.999900,1.728000e+07,-5120,-19.0
57626,IRIS,9R,OBS05,open,18.45662,-81.76946,-4995.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-4928,-67.0
57627,IRIS,9R,OBS06,open,18.45857,-81.74100,-4824.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-4782,-42.0
57628,IRIS,9R,OBS07,open,18.45582,-81.71142,-5075.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-5134,59.0
57629,IRIS,9R,OBS08,open,18.42997,-81.76982,-4367.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-4337,-30.0
57630,IRIS,9R,OBS09,open,18.42769,-81.74040,-4739.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-4565,-174.0
57631,IRIS,9R,OBS10,open,18.42986,-81.71073,-5087.0,EDH_EL1_EL2_ELZ,2022-12-06,2023-06-24 23:59:59.999900,1.736640e+07,-5081,-6.0
57632,IRIS,9R,OBS11,open,18.40188,-81.79697,-2776.0,EDH_EL1_EL2_ELZ,2022-12-07,2023-06-25 23:59:59.999900,1.736640e+07,-2752,-24.0


In [9]:
min_start_dt = ds_9R["start_date"].max()
max_end_dt = ds_9R["end_date"].min()
mean_lon = ds_9R["longitude"].mean()
mean_lat = ds_9R["latitude"].mean()

print(f"9R network: {len(ds_9R)} stations, from {min_start_dt} to {max_end_dt}, mean location: ({mean_lat}, {mean_lon})")

9R network: 36 stations, from 2022-12-09 00:00:00 to 2023-06-23 23:59:59.999900, mean location: (18.37940555555555, -81.75037527777778)


In [10]:
# Save to
root_stage = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\stage_VictorM2"
fpath = os.path.join(root_stage, "data", "stations_9R.csv")
ds_9R.to_csv(fpath, index=False)

# 9R

Réseau 9R situé au Sud des Iles Caïmans, entre la Jamaïque et la côte mexicaine. 

Informations :
* Période disponible : 2022-12-09 au 2023-06-23
* Nombre de capteurs : 39
* Distance médiane inter-capteur : 4.9 km 
* Profondeur : de 2400 à 5080 m 
* Trafic dans la zone : intense 

Le réseau est dense, tous les capteurs ont des données exploitables durant l'année 2023 pour laquelle on dispose des données AIS. 

Bémol : l'étude du bruit ambiant semble particulièrement difficile au vu du trafic intense dans la zone. Tous les spectrogrammes sont dominés par du bruit de bateau. 


La grande partie des capteurs sont exploitables, à l'exception des capteurs suivants : 

* OBS04 (non accessible)
* OBS27 (corrompu)
* OBS32 (non accessible)
* OBS36 (non accessible)

Les passages des navires sont très facilement identifiables et il est a priori facile de trouver une situation favorable. 

## Chargement d'une minute de données pour récupérer les métadonnées utiles 

In [9]:
fmt = "%Y-%m-%d_%H-%M-%S"
start_time = pd.to_datetime("2023-05-03_22-00-00", format=fmt)
end_time = start_time + pd.Timedelta(minutes=1)

print(f"Processing data from {start_time} to {end_time}...")

# ============================================================
# ⚙️ CONFIG
# ============================================================

data_info = {
    "client": "EARTHSCOPE",
    "network": "9R",
    "stations": [
        "OBS01",
        "OBS02",
        "OBS03",
        # "OBS04",
        "OBS05",
        "OBS06",
        "OBS07",
        "OBS08",
        "OBS09",
        "OBS10",
        "OBS11",
        "OBS12",
        "OBS13",
        "OBS14",
        "OBS15",
        "OBS16",
        "OBS17",
        "OBS18",
        "OBS19",
        "OBS20",
        "OBS21",
        "OBS22",
        "OBS23",
        "OBS24",
        "OBS25",
        "OBS26",
        # "OBS27",
        "OBS28",
        "OBS29",
        "OBS30",
        "OBS31",
        # "OBS32",
        "OBS33",
        "OBS34",
        "OBS35",
        # "OBS36",
        "OBS37",
        "OBS38",
        "OBS39",
    ],
    "channel": "EDH",
    "start_time": start_time,
    "end_time": end_time,
}

# ============================================================
# 🚀 RUN PIPELINE
# ============================================================
root_selection_img = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\reseau_9R\img"
)
root_selection_img = os.path.join(root_selection_img, data_info["network"])
root_figures = os.path.join(root_selection_img, f"{start_time.strftime('%Y-%m-%d')}")
os.makedirs(root_figures, exist_ok=True)


st, inv = process_and_plot(
    data_info,
    pre_filt=(0.05, 0.1, 100, 120),
    response_output="ACC",
    save_figures=True,
    save_path=root_figures,
    plot_map=False,
    plot_sig=False,
    plot_spectro=True,
    plot_power_sprectral_density=False,
    show=False,
)


Processing data from 2023-05-03 22:00:00 to 2023-05-03 22:01:00...
Loading data...
Processing traces...

--- 9R.OBS01..EDH ---
SPL: 133.6 dB re 1 µPa

--- 9R.OBS02..EDH ---
SPL: 133.3 dB re 1 µPa

--- 9R.OBS03..EDH ---
SPL: 133.7 dB re 1 µPa

--- 9R.OBS05..EDH ---
SPL: 133.1 dB re 1 µPa

--- 9R.OBS06..EDH ---
SPL: 133.1 dB re 1 µPa

--- 9R.OBS07..EDH ---
SPL: 135.1 dB re 1 µPa

--- 9R.OBS08..EDH ---
SPL: 134.4 dB re 1 µPa

--- 9R.OBS09..EDH ---
SPL: 135.6 dB re 1 µPa

--- 9R.OBS10..EDH ---
SPL: 135.2 dB re 1 µPa

--- 9R.OBS11..EDH ---
SPL: 136.1 dB re 1 µPa

--- 9R.OBS12..EDH ---
SPL: 135.7 dB re 1 µPa

--- 9R.OBS13..EDH ---
SPL: 133.4 dB re 1 µPa

--- 9R.OBS14..EDH ---
SPL: 134.5 dB re 1 µPa

--- 9R.OBS15..EDH ---
SPL: 133.5 dB re 1 µPa

--- 9R.OBS16..EDH ---
SPL: 135.5 dB re 1 µPa

--- 9R.OBS17..EDH ---
SPL: 139.4 dB re 1 µPa

--- 9R.OBS18..EDH ---
SPL: 136.7 dB re 1 µPa

--- 9R.OBS19..EDH ---
SPL: 135.3 dB re 1 µPa

--- 9R.OBS20..EDH ---
SPL: 133.7 dB re 1 µPa

--- 9R.OBS21..EDH ---

In [18]:
# Sauvergarde des coordonnées des capteurs
rcv_id = []
rcv_latitude = []
rcv_longitude = []
rcv_elevation = []
for net in inv:
    for sta in net:
        print(f"Station {sta.code}")
        rcv_id.append(int(sta.code[-2:]))
        print("  Latitude:", sta.latitude)
        rcv_latitude.append(sta.latitude)
        print("  Longitude:", sta.longitude)
        rcv_longitude.append(sta.longitude)
        print("  Elevation:", sta.elevation)
        rcv_elevation.append(sta.elevation)

ds_network_9R = xr.Dataset(
    data_vars=dict(
        latitude=(["receiver_id"], rcv_latitude),
        longitude=(["receiver_id"], rcv_longitude),
        elevation=(["receiver_id"], rcv_elevation),
    ),
    coords=dict(
        receiver_id=rcv_id,
    )
)


fpath = os.path.join(root_data, "9R_network.nc")
ds_network_9R.to_netcdf(fpath)

Station OBS01
  Latitude: 18.51055
  Longitude: -81.76852
  Elevation: -5114.0
Station OBS02
  Latitude: 18.51072
  Longitude: -81.74122
  Elevation: -4533.0
Station OBS03
  Latitude: 18.48431
  Longitude: -81.76879
  Elevation: -5139.0
Station OBS05
  Latitude: 18.45662
  Longitude: -81.76946
  Elevation: -4995.0
Station OBS06
  Latitude: 18.45857
  Longitude: -81.741
  Elevation: -4824.0
Station OBS07
  Latitude: 18.45582
  Longitude: -81.71142
  Elevation: -5075.0
Station OBS08
  Latitude: 18.42997
  Longitude: -81.76982
  Elevation: -4367.0
Station OBS09
  Latitude: 18.42769
  Longitude: -81.7404
  Elevation: -4739.0
Station OBS10
  Latitude: 18.42986
  Longitude: -81.71073
  Elevation: -5087.0
Station OBS11
  Latitude: 18.40188
  Longitude: -81.79697
  Elevation: -2776.0
Station OBS12
  Latitude: 18.40247
  Longitude: -81.76913
  Elevation: -3243.0
Station OBS13
  Latitude: 18.40244
  Longitude: -81.74102
  Elevation: -3652.0
Station OBS14
  Latitude: 18.4018
  Longitude: -81.7116

## Chargement d'une période d'intérêt

In [16]:
fmt = "%Y-%m-%d_%H-%M-%S"
start_time = pd.to_datetime("2023-05-03_22-00-00", format=fmt)
end_time = start_time + pd.Timedelta(hours=4)

In [17]:
# 9R

# end_time = pd.to_datetime("2023-01-02")

print(f"Processing data from {start_time} to {end_time}...")

# ============================================================
# ⚙️ CONFIG
# ============================================================

data_info = {
    "client": "EARTHSCOPE",
    "network": "9R",
    "stations": [
        "OBS01",
        "OBS02",
        "OBS03",
        # "OBS04",
        "OBS05",
        "OBS06",
        "OBS07",
        "OBS08",
        "OBS09",
        "OBS10",
        "OBS11",
        "OBS12",
        "OBS13",
        "OBS14",
        "OBS15",
        "OBS16",
        "OBS17",
        "OBS18",
        "OBS19",
        "OBS20",
        "OBS21",
        "OBS22",
        "OBS23",
        "OBS24",
        "OBS25",
        "OBS26",
        # "OBS27",
        "OBS28",
        "OBS29",
        "OBS30",
        "OBS31",
        # "OBS32",
        "OBS33",
        "OBS34",
        "OBS35",
        # "OBS36",
        "OBS37",
        "OBS38",
        "OBS39",
    ],
    # "stations": [
    #     # "OBS01",
    #     # "OBS02",
    #     # "OBS03",
    #     # # "OBS04",
    #     # "OBS05",
    #     # "OBS06",
    #     # "OBS07",
    #     # "OBS08",
    #     # "OBS09",
    #     # "OBS10",
    #     # "OBS11",
    #     # "OBS12",
    #     # "OBS13",
    #     # "OBS14",
    #     # "OBS15",
    #     # "OBS16",
    #     # "OBS17",
    #     # "OBS18",
    #     "OBS19",
    #     # "OBS20",
    #     # "OBS21",
    #     # "OBS22",
    #     # "OBS23",
    #     # "OBS24",
    #     # "OBS25",
    #     # "OBS26",
    #     # # "OBS27",
    #     # "OBS28",
    #     # "OBS29",
    #     # "OBS30",
    #     # "OBS31",
    #     # # "OBS32",
    #     # "OBS33",
    #     # "OBS34",
    #     # "OBS35",
    #     # # "OBS36",
    #     # "OBS37",
    #     # "OBS38",
    #     # "OBS39",
    # ],
    "channel": "EDH",
    "start_time": start_time,
    "end_time": end_time,
}

# ============================================================
# 🚀 RUN PIPELINE
# ============================================================
root_selection_img = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\reseau_9R\img"
)
root_selection_img = os.path.join(root_selection_img, data_info["network"])
root_figures = os.path.join(
    root_selection_img, f"{start_time.strftime('%Y-%m-%d')}"
)
os.makedirs(root_figures, exist_ok=True)


st, inv = process_and_plot(
    data_info,
    pre_filt=(0.05, 0.1, 100, 120),
    response_output="ACC",
    save_figures=True,
    save_path=root_figures,
    plot_map=False,
    plot_sig=False,
    plot_spectro=True,
    plot_power_sprectral_density=False,
    show=False,
)

for net in inv:
    for sta in net:
        print(f"Station {sta.code}")
        print("  Latitude:", sta.latitude)
        print("  Longitude:", sta.longitude)
        print("  Elevation:", sta.elevation)
        print("  Start:", sta.start_date)
        print("  End:", sta.end_date)

root_wav = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\reseau_9R\data\wav"
# if save:

# Build filename
fmt = "%Y-%m-%d_%H-%M-%S"
date_start = pd.to_datetime(start_time).strftime(fmt)
date_end = (pd.to_datetime(end_time)).strftime(fmt)
fname = f"signal_{data_info['network']}_{data_info['channel']}_{date_start}_{date_end}_"  # on sauvegarde les signaux corriges dans un fichier .wav

for tr in st:
    
    fpath = os.path.join(root_wav, fname+ f"{tr.stats.station}.wav")

    # Save corrected signal
    wavfile.write(
        fpath,
        int(tr.meta.sampling_rate),
        tr.data,
    )

Processing data from 2023-05-03 22:00:00 to 2023-05-04 02:00:00...
Loading data...
Processing traces...

--- 9R.OBS01..EDH ---
SPL: 142.4 dB re 1 µPa

--- 9R.OBS02..EDH ---
SPL: 143.0 dB re 1 µPa

--- 9R.OBS03..EDH ---
SPL: 143.2 dB re 1 µPa

--- 9R.OBS05..EDH ---
SPL: 143.4 dB re 1 µPa

--- 9R.OBS06..EDH ---
SPL: 142.7 dB re 1 µPa

--- 9R.OBS07..EDH ---
SPL: 143.2 dB re 1 µPa

--- 9R.OBS08..EDH ---
SPL: 144.3 dB re 1 µPa

--- 9R.OBS09..EDH ---
SPL: 143.9 dB re 1 µPa

--- 9R.OBS10..EDH ---
SPL: 143.1 dB re 1 µPa

--- 9R.OBS11..EDH ---
SPL: 148.1 dB re 1 µPa

--- 9R.OBS12..EDH ---
SPL: 146.4 dB re 1 µPa

--- 9R.OBS13..EDH ---
SPL: 143.3 dB re 1 µPa

--- 9R.OBS14..EDH ---
SPL: 143.2 dB re 1 µPa

--- 9R.OBS15..EDH ---
SPL: 143.2 dB re 1 µPa

--- 9R.OBS16..EDH ---
SPL: 149.1 dB re 1 µPa

--- 9R.OBS17..EDH ---
SPL: 149.3 dB re 1 µPa

--- 9R.OBS18..EDH ---
SPL: 149.2 dB re 1 µPa

--- 9R.OBS19..EDH ---
SPL: 149.0 dB re 1 µPa

--- 9R.OBS20..EDH ---
SPL: 143.3 dB re 1 µPa

--- 9R.OBS21..EDH ---

In [10]:
for net in inv:
    for sta in net:
        print(f"Station {sta.code}")
        print("  Latitude:", sta.latitude)
        print("  Longitude:", sta.longitude)
        print("  Elevation:", sta.elevation)
        print("  Start:", sta.start_date)
        print("  End:", sta.end_date)

Station OBS01
  Latitude: 18.51055
  Longitude: -81.76852
  Elevation: -5114.0
  Start: 2022-12-05T00:00:00.000000Z
  End: 2023-06-23T23:59:59.999900Z
Station OBS02
  Latitude: 18.51072
  Longitude: -81.74122
  Elevation: -4533.0
  Start: 2022-12-06T00:00:00.000000Z
  End: 2023-06-23T23:59:59.999900Z
Station OBS03
  Latitude: 18.48431
  Longitude: -81.76879
  Elevation: -5139.0
  Start: 2022-12-06T00:00:00.000000Z
  End: 2023-06-23T23:59:59.999900Z
Station OBS05
  Latitude: 18.45662
  Longitude: -81.76946
  Elevation: -4995.0
  Start: 2022-12-06T00:00:00.000000Z
  End: 2023-06-24T23:59:59.999900Z
Station OBS06
  Latitude: 18.45857
  Longitude: -81.741
  Elevation: -4824.0
  Start: 2022-12-06T00:00:00.000000Z
  End: 2023-06-24T23:59:59.999900Z
Station OBS07
  Latitude: 18.45582
  Longitude: -81.71142
  Elevation: -5075.0
  Start: 2022-12-06T00:00:00.000000Z
  End: 2023-06-24T23:59:59.999900Z
Station OBS08
  Latitude: 18.42997
  Longitude: -81.76982
  Elevation: -4367.0
  Start: 2022-12-